<a href="https://colab.research.google.com/github/kazutani2003/EU_M_Math/blob/main/Chap08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

8-1-5　この章で使うライブラリのインポート

In [3]:
#データ加工・処理・分析ライブラリ
import numpy as np
import numpy.random as random
import scipy as sp
from pandas import Series, DataFrame
import pandas as pd

#可視化ライブラリ
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
%matplotlib inline

#機械学習ライブラリ
import sklearn

#少数第３位まで表示
%precision 3

'%.3f'

8-2-1自動車価格データの取り込み

In [33]:
#インポート
import requests, zipfile
import io

#自動車価格データを取得
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'
res = requests.get(url).content

#取得したデータをDataFrameオブジェクトとして読み込み
auto = pd.read_csv(io.StringIO(res.decode('utf-8')), header=None)

#データの列にラベルを設定
auto.columns = ['symboling','normalized-losses','make','fuel-type','aspiration','num-of-doors','body-style','drive-weels','engine-location','wheel-base','length','width','height','curb-weight','engine-type','num-of-cylinders','enine-size','fuel-system','bore','stroke','compression-ratio','horsepower','peak-rpm','city-mpg','highway-mpg','price']
correct_column_names =['symboling','normalized-losses','make','fuel-type','aspiration','num-of-doors','body-style','drive-weels','engine-location','wheel-base','length','width','height','curb-weight','engine-type','num-of-cylinders','enine-size','fuel-system','bore','stroke','compression-ratio','horsepower','peak-rpm','city-mpg','highway-mpg','price']

In [34]:
print('自動車データの形式:{}'.format(auto.shape))

自動車データの形式:(205, 26)


In [35]:
auto.head()

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-weels,engine-location,wheel-base,...,enine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495
1,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500
2,1,?,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500
3,2,164,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950
4,2,164,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450


8-2-2データの整理
8-2-2-1不適切なデータの除去

In [36]:
#それぞれのカラムに？が何個あるかカウント
auto = auto[['price','horsepower','width','height']]
auto.isin(['?']).sum()

,0
price,4
horsepower,2
width,0
height,0


In [37]:
#?をNaNに置換して、NaNがある行を削除
auto = auto.replace('?',np.nan).dropna()
print('自動車のデータの形式:{}',format(auto.shape))

自動車のデータの形式:{} (199, 4)


8-2-2-2型の置換

In [46]:
print('データ型の確認（型変換前）\n{}\n',format(auto.dtypes))

データ型の確認（型変換前）
{}
 price          object
horsepower     object
width         float64
height        float64
dtype: object


In [50]:
auto = auto.assign(price=pd.to_numeric(auto.price))
auto = auto.assign(horsepower=pd.to_numeric(auto.horsepower))
print('データ型の確認（型変換後)\n{}'.format(auto.dtypes))

データ型の確認（型変換後)
price           int64
horsepower      int64
width         float64
height        float64
dtype: object


8-2-2-3相関の確認

In [51]:
auto.corr()

,price,horsepower,width,height
price,1.000000,0.810533,0.753871,0.134990
horsepower,0.810533,1.000000,0.615315,-0.087407
width,0.753871,0.615315,1.000000,0.309223
height,0.134990,-0.087407,0.309223,1.000000


8-2-3モデル構築と評価
8-2-4モデル構築とモデル評価の流れのまとめ
練習問題8-1

In [60]:
#データ分割（訓練データとテストデータ）のためのインポート
from sklearn.model_selection import train_test_split

#重回帰のモデル構築のためのインポート
from sklearn.linear_model import LinearRegression

#目的変数にpriceを指定、説明変数にそれ以外を指定
x = auto.drop('price',axis=1)
y = auto['price']

#訓練データとテストデータに分ける
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.5,random_state=0)

#重回帰クラスの初期化と学習
model = LinearRegression()
model.fit(x_train,y_train)

#決定係数を表示
print('決定係数(train):{:.3f}'.format(model.score(x_train,y_train)))
print('決定係数(test):{:.3f}'.format(model.score(x_test,y_test)))

#回帰係数と切符を表示
print('\n回帰係数\n{}'.format(pd.Series(model.coef_, index=x.columns)))
print('切符:{:.3f}'.format(model.intercept_))

決定係数(train):0.733
決定係数(test):0.737

回帰係数
horsepower      81.651078
width         1829.174506
height         229.510077
dtype: float64
切符:-128409.046
